In [1]:
import sys
sys.path.insert(0, r"PROJECT_ROOT")

import pandas as pd
import numpy as np
from IPython.display import display, HTML
from datetime import datetime

print("✅ 环境初始化完成")

✅ 环境初始化完成


In [3]:
from openai import OpenAI

# ⚠️ 请替换成你的真实 DeepSeek Key
DS_KEY = "your_deepseek_api_key"

client = OpenAI(
    api_key=DS_KEY,
    base_url="https://api.deepseek.com"
)

# 测试连通性
try:
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": "回复：OK"}],
        max_tokens=10
    )
    print("✅ DeepSeek 连通成功")
except Exception as e:
    print("❌ DeepSeek 连接失败:", e)

✅ DeepSeek 连通成功


In [4]:
from ts_init import fetch_stock_data
from indicators import IndicatorFactory

# 测试单股票
df = fetch_stock_data("600519.SH")
print(f"✅ 数据加载成功，共 {len(df)} 行")
df.head()

✅ AStockAgent 数据引擎已就绪
✅ 指标工厂加载成功
✅ 从缓存加载 600519.SH，共 339 行
✅ 数据加载成功，共 339 行


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,600519.SH,1970-01-01 00:00:00.020260601,1327.00,1327.00,1301.31,1309.60,1326.00,-16.40,-1.2368,43844.60,5.741133e+06
1,600519.SH,1970-01-01 00:00:00.020260529,1270.60,1329.00,1270.00,1326.00,1275.98,50.02,3.9201,76478.05,1.003739e+07
2,600519.SH,1970-01-01 00:00:00.020260528,1290.00,1304.00,1271.00,1275.98,1303.00,-27.02,-2.0737,45889.98,5.895475e+06
3,600519.SH,1970-01-01 00:00:00.020260527,1268.02,1319.00,1250.10,1303.00,1273.38,29.62,2.3261,82727.91,1.058657e+07
4,600519.SH,1970-01-01 00:00:00.020260526,1285.35,1289.89,1270.01,1273.38,1285.88,-12.50,-0.9721,45931.62,5.867831e+06


In [5]:
# ---------- 1. 构建股票摘要 ----------
def build_stock_summary(df, ts_code="600519.SH", position_ratio=None):
    df = df.dropna(subset=["close", "ma5", "ma20", "rsi14"])
    if len(df) < 2:
        return "数据不足"
    
    L = df.iloc[-1]
    prev = df.iloc[-2]
    
    trend = "多头" if L["ma5"] > L["ma20"] else "空头"
    rsi_zone = "超买" if L["rsi14"] > 70 else ("超卖" if L["rsi14"] < 30 else "中性")
    
    lines = [
        f"股票：{ts_code}",
        f"收盘价：{L['close']:.2f} 元（前一日 {prev['close']:.2f}）",
        f"均线：MA5={L['ma5']:.1f} / MA20={L['ma20']:.1f} → {trend}",
        f"RSI：{L['rsi14']:.1f} → {rsi_zone}"
    ]
    if position_ratio:
        lines.append(f"持仓占比：{position_ratio*100:.0f}%")
    return "\n".join(lines)

# ---------- 2. 历史回测概率 ----------
def backtest_probability(df):
    df = df.dropna(subset=["close", "ma5", "ma20"]).copy()
    if len(df) < 100:
        return "样本不足"
    
    df["future_ret_5d"] = df["close"].pct_change(5).shift(-5)
    cond = (df["ma5"] > df["ma20"]) & (df["rsi14"].between(30, 70))
    hist = df[cond]
    
    if len(hist) < 10:
        return "样本不足"
    
    win_rate = (hist["future_ret_5d"] > 0).mean() * 100
    return f"📊 过去5年出现类似形态 {len(hist)} 次，未来5天上涨概率 {win_rate:.1f}%"

# ---------- 3. AI 建议 ----------
def ask_deepseek_advice(summary_text):
    system_msg = {
        "role": "system",
        "content": "你是A股实战派投资顾问，只基于技术指标给建议，不编造消息。"
    }
    user_msg = {
        "role": "user",
        "content": f"{summary_text}\n\n请给出：1）技术面定性 2）操作建议 3）风控提醒"
    }
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[system_msg, user_msg],
        temperature=0.5,
        max_tokens=350
    )
    return resp.choices[0].message.content

# ---------- 4. 渲染卡片 ----------
def render_card(summary, advice, prob):
    html = f"""
    <div style="
        border: 1px solid #ddd;
        border-radius: 10px;
        padding: 15px;
        background-color: #f9f9f9;
        font-family: 'Microsoft YaHei', sans-serif;
        max-width: 800px;
    ">
        <h2>📊 持仓体检报告</h2>
        <hr>
        <pre style="font-size: 14px; background: #fff; padding: 10px; border-radius: 5px;">{summary}</pre>
        <p><b>🤖 AI 建议</b><br>{advice}</p>
        <p><b>📈 历史回测</b><br>{prob}</p>
        <p style="color: #666; font-size: 12px;">⚠️ 以上仅为技术指标参考，不构成买卖依据</p>
    </div>
    """
    display(HTML(html))

print("✅ 工具函数定义完成")

✅ 工具函数定义完成


In [6]:
def run_full_check(ts_code="600519.SH", position_ratio=0.60):
    print(f"🔄 正在分析 {ts_code}...")
    
    # 1. 拉取并计算指标
    df = fetch_stock_data(ts_code)
    df = IndicatorFactory.add_all_indicators(df.copy())
    
    # 2. 生成各部分内容
    summary = build_stock_summary(df, ts_code, position_ratio)
    prob = backtest_probability(df)
    advice = ask_deepseek_advice(summary)
    
    # 3. 渲染报告
    render_card(summary, advice, prob)
    print("\n✅ 体检完成")

# 测试
run_full_check("600519.SH", position_ratio=0.60)

🔄 正在分析 600519.SH...
✅ 从缓存加载 600519.SH，共 339 行



✅ 体检完成
